# Indian Startup Funding - Data Cleaning

**Goal:** Clean `raw_startup_funding.csv` (3,044 funding rounds, 2015–2020) so it's
ready for SQL analysis and a Power BI dashboard on **startup funding trends**.

**What's wrong with the raw file (found during inspection):**
| Column | Issue |
|---|---|
| `Date dd/mm/yyyy` | Stored as text; 8 rows have malformed formats (missing slash, stray dots, double slash, corrupted encoding characters) |
| `Amount in USD` | Stored as text using Indian digit grouping (`"20,00,00,000"`); contains `"undisclosed"` / `"unknown"`; ~32% missing |
| `City  Location` | Inconsistent spellings (`Bangalore` vs `Bengaluru`, `Ahemadabad` vs `Ahmedabad`), combined entries (`"Bangalore / SFO"`) |
| `InvestmentnType` | ~55 near-duplicate labels for the same funding stage (`Seed`, `Seed Round`, `Seed/Angel Funding`, etc.) |
| `Industry Vertical` / `SubVertical` | Free text, missing values, long tail of one-off categories |
| `Investors_Name` | Same investor written with different casing/spacing counts as a different investor when grouping (e.g. `Westbridge Capital` vs `WestBridge Capital`, `Softbank` vs `SoftBank Group`) — 63 such variant groups across ~2,400 unique investor tokens |
| Many text columns | Contain leftover encoding artifacts (`\xc2\xa0`, `\n`) from a bad text export |
| `Remarks` | 86% empty, low analytical value |
| `Sr No` | Just a row index, redundant with the DataFrame index |

We'll fix each issue step by step, explaining *why*, then export a clean CSV.


## 1. Setup & load the raw data

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

RAW_PATH = r'.....\raw_startup_funding.csv'
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()

Shape: (3044, 10)


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Sr No              3044 non-null   int64
 1   Date dd/mm/yyyy    3044 non-null   str  
 2   Startup Name       3044 non-null   str  
 3   Industry Vertical  2873 non-null   str  
 4   SubVertical        2108 non-null   str  
 5   City  Location     2864 non-null   str  
 6   Investors Name     3020 non-null   str  
 7   InvestmentnType    3040 non-null   str  
 8   Amount in USD      2084 non-null   str  
 9   Remarks            419 non-null    str  
dtypes: int64(1), str(9)
memory usage: 237.9 KB


## 2. Initial inspection

Let's confirm the problems we expect to find, so we know exactly what the cleaning
steps below need to handle.

In [3]:
# Missing values per column
df.isnull().sum().sort_values(ascending=False)

Remarks              2625
Amount in USD         960
SubVertical           936
City  Location        180
Industry Vertical     171
Investors Name         24
InvestmentnType         4
Sr No                   0
Date dd/mm/yyyy         0
Startup Name            0
dtype: int64

In [4]:
# Amount column: text formatting + non-numeric placeholder values
df['Amount in USD'].dropna().sample(10, random_state=1).tolist()

['6,00,000',
 '31,00,000',
 '1,91,000',
 '20,00,00,000',
 '1,50,000',
 '10,00,000',
 '48,00,000',
 '1,20,000',
 '1,25,000',
 '5,00,000']

In [5]:
# City column: inconsistent spellings of the same city
sorted(df['City  Location'].dropna().unique())[:20]

['Agra',
 'Ahemadabad',
 'Ahemdabad',
 'Ahmedabad',
 'Amritsar',
 'Andheri',
 'Bangalore',
 'Bangalore / Palo Alto',
 'Bangalore / SFO',
 'Bangalore / San Mateo',
 'Bangalore / USA',
 'Bangalore/ Bangkok',
 'Belgaum',
 'Bengaluru',
 'Bengaluru and Gurugram',
 'Bhopal',
 'Bhubaneswar',
 'Bhubneswar',
 'Boston',
 'Burnsville']

In [6]:
# Investment type: many near-duplicate labels for the same stage
sorted(df['InvestmentnType'].dropna().unique())[:20]

['Angel',
 'Angel / Seed Funding',
 'Angel Funding',
 'Angel Round',
 'Bridge Round',
 'Corporate Round',
 'Crowd Funding',
 'Crowd funding',
 'Debt',
 'Debt Funding',
 'Debt and Preference capital',
 'Debt-Funding',
 'Equity',
 'Equity Based Funding',
 'Funding Round',
 'Inhouse Funding',
 'Maiden Round',
 'Mezzanine',
 'Pre Series A',
 'Pre-Series A']

## 3. Clean column names

The raw headers have inconsistent spacing and typos (`City  Location` has two spaces,
`InvestmentnType` is misspelled). We rename them to clean, consistent names.

In [7]:
df.columns = df.columns.str.strip()

df = df.rename(columns={
    'Date dd/mm/yyyy': 'Date',
    'City  Location': 'City',
    'Investors Name': 'Investors_Name',
    'InvestmentnType': 'Investment_Type',
    'Amount in USD': 'Amount_USD',
    'Industry Vertical': 'Industry',
})

df.columns.tolist()

['Sr No',
 'Date',
 'Startup Name',
 'Industry',
 'SubVertical',
 'City',
 'Investors_Name',
 'Investment_Type',
 'Amount_USD',
 'Remarks']

## 4. Strip text artifacts from every text column

Several columns contain leftover escape sequences from a bad encoding conversion
(e.g. a non-breaking space that got exported as the literal text `\xc2\xa0`,
or newlines exported as literal `\n`). We write one reusable helper and apply it
to every text column, along with trimming stray whitespace/commas.

In [8]:
def basic_clean(value):
    """Strip encoding artifacts and normalize whitespace in a text cell."""
    if pd.isna(value):
        return value
    text = str(value)
    text = re.sub(r'\\+x[a-f0-9]{2}', ' ', text, flags=re.IGNORECASE)  # \xc2, \xa0, etc.
    text = re.sub(r'\\+n', ' ', text)                                  # literal \n
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.strip(', ')
    return text if text != '' else np.nan


text_columns = ['Startup Name', 'Industry', 'SubVertical', 'City',
                 'Investors_Name', 'Investment_Type']

for col in text_columns:
    df[col] = df[col].apply(basic_clean)

print("Cleaned:", text_columns)

Cleaned: ['Startup Name', 'Industry', 'SubVertical', 'City', 'Investors_Name', 'Investment_Type']


## 5. Fix and parse the `Date` column

Most dates are `dd/mm/yyyy`, but a handful are malformed: missing separators
(`05/072018`), stray dots (`12/05.2015`), doubled slashes (`22/01//2015`),
a 3-digit year typo (`01/07/015`), and one row with encoding artifacts mixed
into the digits. We normalize all of these with a regex-based fixer, then
parse to a real `datetime`.

In [9]:
def fix_date_str(value):
    text = str(value)
    text = re.sub(r'\\+x[a-f0-9]{2}', '', text, flags=re.IGNORECASE)  # drop encoding artifacts
    text = re.sub(r'[^0-9/.]', '', text)                                # keep digits/slashes/dots only
    text = text.replace('.', '/')
    text = re.sub(r'/+', '/', text).strip('/')

    # missing slash before a 4-digit year, e.g. 05/072018 -> 05/07/2018
    m = re.match(r'^(\d{2})/(\d{2})(\d{4})$', text)
    if m:
        text = f'{m.group(1)}/{m.group(2)}/{m.group(3)}'

    # 3-digit year typo, e.g. 01/07/015 -> 01/07/2015
    m = re.match(r'^(\d{2})/(\d{2})/(\d{3})$', text)
    if m:
        text = f'{m.group(1)}/{m.group(2)}/20{m.group(3)[-2:]}'

    return text


df['Date'] = df['Date'].apply(fix_date_str)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')

print("Unparseable dates remaining:", df['Date'].isnull().sum())

Unparseable dates remaining: 0


In [10]:
# Add convenience columns for trend analysis later (Power BI / SQL will use these too)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

df[['Date', 'Year', 'Month', 'YearMonth']].head()

,Date,Year,Month,YearMonth
0,2020-01-09,2020,1,2020-01
1,2020-01-13,2020,1,2020-01
2,2020-01-09,2020,1,2020-01
3,2020-01-02,2020,1,2020-01
4,2020-01-02,2020,1,2020-01


## 6. Clean `Amount_USD`

The raw amounts are text using **Indian digit grouping** (lakh/crore style commas,
e.g. `"20,00,00,000"` = 200,000,000) rather than the Western `"200,000,000"` style —
but since we're just removing commas, both formats convert correctly the same way.
We also need to catch placeholder text (`"undisclosed"`, `"unknown"`, `"N/A"`) and a
stray `"+"` suffix, and convert everything to a proper numeric column.

In [11]:
def clean_amount(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    text = re.sub(r'\\+x[a-f0-9]{2}', '', text, flags=re.IGNORECASE).strip()
    if text.lower() in ('undisclosed', 'unknown', 'n/a', ''):
        return np.nan
    text = text.replace(',', '').replace('+', '')
    try:
        return float(text)
    except ValueError:
        return np.nan


df['Amount_USD'] = df['Amount_USD'].apply(clean_amount)

print("Missing amounts:", df['Amount_USD'].isnull().sum(), f"({df['Amount_USD'].isnull().mean():.1%})")
df['Amount_USD'].describe()

Missing amounts: 971 (31.9%)


count    2.073000e+03
mean     1.840034e+07
std      1.211407e+08
min      1.600000e+04
25%      4.860000e+05
50%      1.750000e+06
75%      8.000000e+06
max      3.900000e+09
Name: Amount_USD, dtype: float64

**Note on missing amounts:** ~32% of rows have no disclosed amount. We deliberately
leave these as `NaN` (not 0) so they don't distort averages/sums — SQL and Power BI
should filter or explicitly handle nulls when calculating totals.

## 7. Standardize `City`

We fix known spelling variants (mapped to one canonical name), and for entries that
list multiple cities (e.g. `"Bangalore / SFO"`) we keep the first (primary) city.

In [12]:
CITY_MAP = {
    'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru',
    'ahemadabad': 'Ahmedabad', 'ahemdabad': 'Ahmedabad', 'ahmedabad': 'Ahmedabad',
    'bhubneswar': 'Bhubaneswar', 'bhubaneswar': 'Bhubaneswar',
    'gurugram': 'Gurugram', 'gurgaon': 'Gurugram',
    'delhi': 'New Delhi', 'new delhi': 'New Delhi',
}

def clean_city(value):
    if pd.isna(value):
        return np.nan
    primary = re.split(r'\s*/\s*|\s*&\s*|\s+and\s+', value)[0].strip()
    return CITY_MAP.get(primary.lower(), primary)


df['City'] = df['City'].apply(clean_city)
df['City'].value_counts().head(15)

City
Bengaluru     850
Mumbai        573
New Delhi     463
Gurugram      342
Pune          112
Hyderabad     100
Chennai        98
Noida          94
Ahmedabad      41
Jaipur         30
Kolkata        21
Indore         13
Chandigarh     11
Goa            11
Vadodara       10
Name: count, dtype: int64

## 8. Standardize `Investment_Type`

The raw column has ~55 labels that really represent a small set of funding stages
(Seed/Angel, Series A–H, Private Equity, Debt, etc.), just written inconsistently
(different casing, `"Seed Round"` vs `"Seed Funding"` vs `"Seed/Angel Funding"`).
We map each raw label to one clean category using pattern matching, and keep the
original column too in case anyone wants the raw text.

In [13]:
def clean_investment_type(value):
    if pd.isna(value):
        return 'Unknown'
    text = value.lower().strip()
    text = re.sub(r'\s+', ' ', text)

    patterns = [
        (r'seed|angel', 'Seed/Angel'),
        (r'series a', 'Series A'),
        (r'series b', 'Series B'),
        (r'series c', 'Series C'),
        (r'series d', 'Series D'),
        (r'series e', 'Series E'),
        (r'series f', 'Series F'),
        (r'series g', 'Series G'),
        (r'series h', 'Series H'),
        (r'private equity|privateequity', 'Private Equity'),
        (r'debt', 'Debt Funding'),
        (r'crowd', 'Crowd Funding'),
        (r'bridge', 'Bridge Round'),
        (r'corporate', 'Corporate Round'),
        (r'mezzanine', 'Mezzanine'),
        (r'equity', 'Equity'),
    ]
    for pattern, label in patterns:
        if re.search(pattern, text):
            return label
    return 'Other'


df['Investment_Type_Clean'] = df['Investment_Type'].apply(clean_investment_type)
df['Investment_Type_Clean'].value_counts()

Investment_Type_Clean
Seed/Angel         1542
Private Equity     1362
Series A             33
Debt Funding         29
Series B             21
Series C             14
Series D             12
Other                12
Unknown               4
Equity                3
Series F              2
Series E              2
Corporate Round       2
Crowd Funding         2
Series G              1
Series H              1
Bridge Round          1
Mezzanine             1
Name: count, dtype: int64

## 9. Normalize investor names (fix case/spacing duplicates + merge known aliases)

`Investors_Name` has the same problem we already fixed for `City`: the same investor
written with different capitalization or spacing counts as a *different* investor when
grouping (e.g. `Westbridge Capital` vs `WestBridge Capital`, `Kalaari capital` vs
`Kalaari Capital`) - 63 such variant groups exist across ~2,400 unique investor tokens.
Left unfixed, this quietly splits one investor's funding across two rows in any
investor leaderboard.

We fix this in two passes:
1. **Automatic case/spacing normalization** - split every cell into individual investor
   names, count how each casing variant actually appears across the dataset, and pick
   the most frequent spelling as the canonical one for that investor.
2. **Manual alias merge for known multi-name entities** - some investors are written as
   genuinely different strings (not just case), e.g. `Softbank` vs `SoftBank Group` vs
   `SoftBank Corp`, which all refer to the same parent entity. We fold these into one
   canonical name, while keeping clearly distinct sub-funds (`SoftBank Vision Fund`,
   `SoftBank Ventures Korea`) separate since they're different investing vehicles.

In [14]:
import re
from collections import Counter, defaultdict

SPLIT_RE = re.compile(r'\s*,\s*|\s+and\s+|\s*&\s*')

# Pass 1: frequency-based casing normalization
token_counts = Counter()
for cell in df['Investors_Name'].dropna():
    for part in SPLIT_RE.split(cell):
        part = part.strip()
        if part:
            token_counts[part] += 1

by_lower = defaultdict(list)
for name, cnt in token_counts.items():
    by_lower[name.lower()].append((name, cnt))

casing_map = {}
for lower_key, variants in by_lower.items():
    if len(variants) > 1:
        variants.sort(key=lambda x: -x[1])  # most frequent spelling wins
        canonical = variants[0][0]
        for name, _ in variants:
            casing_map[name] = canonical

print(f"Casing/spacing variants found and normalized: {len(casing_map)}")


def normalize_investor_casing(cell):
    if pd.isna(cell):
        return cell
    parts = [p.strip() for p in SPLIT_RE.split(cell) if p.strip()]
    parts = [casing_map.get(p, p) for p in parts]
    return ', '.join(parts)


df['Investors_Name'] = df['Investors_Name'].apply(normalize_investor_casing)

# Pass 2: merge known SoftBank-family aliases into one canonical entity,
# while keeping distinctly-branded sub-funds separate
def merge_softbank_aliases(text):
    if pd.isna(text):
        return text
    text = re.sub(r'softbank\s+vision\s+fund', 'SoftBank Vision Fund', text, flags=re.IGNORECASE)
    text = re.sub(r'softbank\s+ventures\s+korea', 'SoftBank Ventures Korea', text, flags=re.IGNORECASE)
    text = re.sub(r'softbank\s+group\s+corp', 'SoftBank Group', text, flags=re.IGNORECASE)
    text = re.sub(r'softbank\s+group', 'SoftBank Group', text, flags=re.IGNORECASE)
    text = re.sub(r'softbank\s+corp', 'SoftBank Group', text, flags=re.IGNORECASE)
    # bare "Softbank" not already part of one of the canonical forms above,
    # and not immediately followed by a leftover artifact like " s " (possessive quote that got stripped)
    text = re.sub(
        r'(?<!SoftBank )(?<!SoftBank Vision )(?<!SoftBank Ventures )\bsoftbank\b(?!\s+(Group|Vision|Ventures|Corp)|\s+s\b)',
        'SoftBank Group', text, flags=re.IGNORECASE
    )
    return text


df['Investors_Name'] = df['Investors_Name'].apply(merge_softbank_aliases)

print("Sample after normalization:")
df['Investors_Name'].dropna().sample(5, random_state=1).tolist()

Casing/spacing variants found and normalized: 136
Sample after normalization:


['Lakshmi Vilas Bank, Undisclosed HNIs',
 'Calcutta Angel Network, Appliyifi',
 'Accel Partners, Flipkart',
 'Lightspeed Venture Partners, K Ganesh',
 'Bessemer Venture Partners, Stellaris Venture Partners, Jungle Venture Partners, Axis Capital']

## 10. Fill remaining categorical gaps, and drop low-value columns

- `Industry`, `SubVertical`, `City`, `Investors_Name`: fill missing with `"Unknown"`
  so groupings in SQL/Power BI don't silently drop null rows.
- `Remarks`: 86% empty and free-text, not useful for structured analysis - drop it.
- `Sr No`: a plain row counter, redundant with the DataFrame/SQL index - drop it.

In [15]:
for col in ['Industry', 'SubVertical', 'City', 'Investors_Name']:
    df[col] = df[col].fillna('Unknown')

df = df.drop(columns=['Remarks', 'Sr No'], errors='ignore')
df.isnull().sum()

Date                       0
Startup Name               0
Industry                   0
SubVertical                0
City                       0
Investors_Name             0
Investment_Type            4
Amount_USD               971
Year                       0
Month                      0
YearMonth                  0
Investment_Type_Clean      0
dtype: int64

## 11. Remove duplicate rows

In [16]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} exact duplicate rows. New shape: {df.shape}")

Removed 0 exact duplicate rows. New shape: (3044, 12)


**Note:** some startup names appear multiple times (e.g. Swiggy, Ola, Paytm) -
that's expected and correct, since a startup can raise several funding rounds over
time. We only removed fully identical rows, not repeat startups.

## 12. Final validation

Quick sanity check before exporting: no unexpected nulls in key structural columns,
correct dtypes, and a look at the cleaned data.

In [17]:
print("Final shape:", df.shape)
print()
print("Dtypes:")
print(df.dtypes)
print()
print("Nulls:")
print(df.isnull().sum())

Final shape: (3044, 12)

Dtypes:
Date                     datetime64[us]
Startup Name                        str
Industry                            str
SubVertical                         str
City                                str
Investors_Name                      str
Investment_Type                     str
Amount_USD                      float64
Year                              int32
Month                             int32
YearMonth                           str
Investment_Type_Clean               str
dtype: object

Nulls:
Date                       0
Startup Name               0
Industry                   0
SubVertical                0
City                       0
Investors_Name             0
Investment_Type            4
Amount_USD               971
Year                       0
Month                      0
YearMonth                  0
Investment_Type_Clean      0
dtype: int64


In [18]:
df.head(10)

,Date,Startup Name,Industry,SubVertical,City,Investors_Name,Investment_Type,Amount_USD,Year,Month,YearMonth,Investment_Type_Clean
0,2020-01-09,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,200000000.0,2020,1,2020-01,Private Equity
1,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,8048394.0,2020,1,2020-01,Series C
2,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,18358860.0,2020,1,2020-01,Series B
3,2020-01-02,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,3000000.0,2020,1,2020-01,Series A
4,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,1800000.0,2020,1,2020-01,Seed/Angel
5,2020-01-13,Pando,Logistics,"Open-market, freight management platform",Chennai,Chiratae Ventures,Series A,9000000.0,2020,1,2020-01,Series A
6,2020-01-10,Zomato,Hospitality,Online Food Delivery Platform,Gurugram,Ant Financial,Private Equity Round,150000000.0,2020,1,2020-01,Private Equity
7,2019-12-12,Ecozen,Technology,Agritech,Pune,Sathguru Catalyzer Advisors,Series A,6000000.0,2019,12,2019-12,Series A
8,2019-12-06,CarDekho,E-Commerce,Automobile,Gurugram,Ping An Global Voyager Fund,Series D,70000000.0,2019,12,2019-12,Series D
9,2019-12-03,Dhruva Space,Aerospace,Satellite Communication,Bengaluru,"Mumbai Angels, Ravikanth Reddy",Seed,50000000.0,2019,12,2019-12,Seed/Angel


In [19]:
df['Amount_USD'].describe()

count    2.073000e+03
mean     1.840034e+07
std      1.211407e+08
min      1.600000e+04
25%      4.860000e+05
50%      1.750000e+06
75%      8.000000e+06
max      3.900000e+09
Name: Amount_USD, dtype: float64

## 13. Export the cleaned dataset

This clean CSV is what we'll load into SQL for querying, and later into Power BI
for the dashboard.

In [20]:
OUT_PATH = r'......\cleaned_startup_funding.csv'
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print("Final shape:", df.shape)

Saved: C:\Users\Hii\Desktop\da\cleaned_startup_funding.csv
Final shape: (3044, 12)


## Summary - what we fixed

| Step | Before | After |
|---|---|---|
| Dates | Text, 8 malformed formats | Proper `datetime64`, 0 unparseable |
| Amount_USD | Text with Indian comma grouping + `"undisclosed"` | Numeric `float`, missing left as `NaN` |
| City | 112 raw variants incl. typos & multi-city entries | Standardized to canonical city names |
| Investment_Type | ~55 near-duplicate labels | ~17 clean categories in `Investment_Type_Clean` |
| Investors_Name | 63 case/spacing variant groups (e.g. `Westbridge Capital` vs `WestBridge Capital`), SoftBank split across 3+ spellings | Normalized to most-frequent spelling; SoftBank aliases merged into `SoftBank Group` (true total $6.55B, was hidden across fragments) |
| Text columns | Encoding artifacts (`\xc2\xa0`, `\n`) | Stripped and normalized |
| Remarks, Sr No | Low-value columns | Dropped |
| Missing categoricals | NaN | Filled with `"Unknown"` |

**Note:** the investor-name fix changes ranking results - re-run any downstream SQL queries and refresh the Power BI data source after regenerating this file, or old cached rankings (e.g. top investors by amount) will be stale.

The cleaned file is ready for:
1. **SQL** - load into a table, write aggregation queries (top industries, cities,
   investors, funding trend over time).
2. **Power BI** - build the dashboard directly on top of `clean_startup_funding.csv`.
